In [ ]:
import pandas as pd

# --- Load datasets ---
df1 = pd.read_csv('/content/ParsedResale_updatedSept2025.csv')
df2 = pd.read_csv('/content/touchable_resale_final_round.csv')

# --- Normalize column names for consistency ---
df1.columns = df1.columns.str.strip().str.title()
df2.columns = df2.columns.str.strip().str.title()

# --- Key columns for uniqueness check ---
key_cols = ['Town', 'Development', 'Address', 'Unit Number']

# --- Create temporary normalized keys (preserve originals) ---
df1['_property_key'] = (
    df1[key_cols]
    .astype(str)
    .apply(lambda x: x.str.strip().str.lower())
    .agg('|'.join, axis=1)
)
df2['_property_key'] = (
    df2[key_cols]
    .astype(str)
    .apply(lambda x: x.str.strip().str.lower())
    .agg('|'.join, axis=1)
)

# --- Identify unique property keys ---
unique_df1_keys = df1.loc[~df1['_property_key'].isin(df2['_property_key']), '_property_key']
unique_df2_keys = df2.loc[~df2['_property_key'].isin(df1['_property_key']), '_property_key']

# --- Extract full rows (with all columns) corresponding to those keys ---
only_in_df1 = df1[df1['_property_key'].isin(unique_df1_keys)].copy()
only_in_df2 = df2[df2['_property_key'].isin(unique_df2_keys)].copy()

print(f"🔹 Listings only in Dataset 1: {len(only_in_df1)}")
print(f"🔹 Listings only in Dataset 2: {len(only_in_df2)}")

# --- Drop helper columns before saving ---
only_in_df1.drop(columns=['_property_key'], inplace=True)
only_in_df2.drop(columns=['_property_key'], inplace=True)

# --- Save unique subsets for review (with full columns) ---
only_in_df1.to_csv('unique_to_dataset1.csv', index=False)
only_in_df2.to_csv('unique_to_dataset2.csv', index=False)

print("✅ Saved full unique entries (including Age Restricted, Bedrooms, Transaction Start Date, etc.)")


🔹 Listings only in Dataset 1: 8
🔹 Listings only in Dataset 2: 0
✅ Saved full unique entries (including Age Restricted, Bedrooms, Transaction Start Date, etc.)
